In [1]:
import numpy as np
import pandas as pd

path = '/kaggle/input/'

def read_csv(a,p=path):
    return pd.read_csv(p+a, sep='\t', header=None, names=['protein','go_term','score'])

In [2]:
model_A = read_csv('cafa-6-blend-goa-negative-propagation/submission.tsv')
# ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~                       
model_A['key']   = model_A['protein'] + '|' + model_A['go_term']
model_A['score'] = model_A['score'].clip(0, 1.0)

model_B = read_csv('cafa-6-protein-function-prediction-with-1d-cnn/submission.tsv')
# ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~
model_B['key']   = model_B['protein'] + '|' + model_B['go_term']
model_B['score'] = model_B['score'].clip(0, 1.0)

model_C = read_csv('cafa-6-protein-function-starter-eda-model/submission.tsv')
# ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~
model_C['key']   = model_C['protein'] + '|' + model_C['go_term']
model_C['score'] = model_C['score'].clip(0, 1.0)

In [3]:
feats = ['protein','go_term','score']

- original . 1 . LB = 0.355, &nbsp; models = [ A, B ], &nbsp; weights = [ 0.80 + 0.20 ]
- original . 2 . LB = 0.345, &nbsp; models = [ A, C ], &nbsp; weights = [ 0.96 + 0.04 ]

In [4]:
NOISE_L, NOISE_R = 0.0015,0.0015  # originals =  noise  noise_absent

weight_model_A   = 0.80           # originals =  0.80   0.96
weight_model_B   = 0.09           # originals =  0.20    -
weight_model_C   = 0.01           # originals =   -     0.04

DIVERSITY_BONUS  = 0.04           # originals = 0.05,   0.08
CONF_THRESHOLD   = 0.007          # originals = 0.01,   0.013

dict_A = dict(zip(model_A['key'], model_A['score']))
dict_B = dict(zip(model_B['key'], model_B['score']))
dict_C = dict(zip(model_C['key'], model_C['score']))

all_keys =\
    set(model_A['key'].unique()) | \
    set(model_B['key'].unique()) | \
    set(model_C['key'].unique()) 

results = []
for key in all_keys:
    protein, go_term = key.split('|')
    score_A, score_B, score_C, = dict_A.get(key,0), dict_B.get(key,0), dict_C.get(key,0),
    
    weighted_score = \
        score_A * weight_model_A + \
        score_B * weight_model_B + \
        score_C * weight_model_C
    
    if score_A > 0 and score_B > 0 and score_C > 0:
        final_score = min(weighted_score + DIVERSITY_BONUS, 1.0)
    else:
        final_score = weighted_score

    final_score += np.random.uniform(-NOISE_L, NOISE_R)
    final_score  = np.clip(final_score, 0, 1)

    results.append(dict(zip(feats,[protein, go_term, final_score])))

ensemble = pd.DataFrame(results)

In [5]:
ensemble = ensemble.sort_values('score', ascending=False)
                                                              
ensemble = ensemble[ensemble['score'] >= CONF_THRESHOLD]

ensemble[ feats ].to_csv('submission.tsv', sep='\t', index=False, header=False)